In [ ]:
import os
import os.path as op
import pickle
import sys
import warnings
from collections import defaultdict
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import permutation_test, spearmanr, ttest_rel
from sklearn.exceptions import InconsistentVersionWarning
from sklearn.linear_model import LinearRegression

sys.path.append(op.abspath(op.join(__file__, "../../../..")))
from utils.utils import correlation_score

sns.set_style("ticks")
sns.set_context("talk", font_scale=1.2, rc={"axes.labelpad": 10})
pd.set_option("display.float_format", "{:.3}".format)

warnings.filterwarnings("ignore", category=InconsistentVersionWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", "{:.3}".format)

In [ ]:
ABS_PATH = sys.path[-1]
RESULTS_PATH = op.join(
    ABS_PATH, "validation/3_prediction/3.1_brain_age_prediction/results"
)
FIG_DIR = op.join(ABS_PATH, "validation/3_prediction/3.1_brain_age_prediction/figures")
os.makedirs(FIG_DIR, exist_ok=True)

PALETTE = {
    "Actual": "#283F94",
    "Predicted": "#AE3033",
}

CONTRASTS = (
    "REST",
    "EMOTION FACES-SHAPES",
    "GAMBLING REWARD",
    "LANGUAGE MATH-STORY",
    "RELATIONAL REL",
    "SOCIAL TOM-RANDOM",
    "WM 2BK-0BK",
    "MOTOR AVG",
)

DATASETS = ["ukb_actual", "ukb_predicted"]
ALL_TARGETS = pd.read_csv(
    op.join(ABS_PATH, "validation/3_prediction/__ukb_phenotypes/ukb_all_targets.py")
)
ALL_TARGETS = ALL_TARGETS[ALL_TARGETS["age"].notna()]

In [ ]:
## HELPER FUNCTIONS
def correct_braingap(
    actual_age, predicted_age, actual_age_test=None, predicted_age_test=None
):
    """
    Function to correct the predicted age and calculate the age gap.

    Parameters
    ----------
    actual_age : np.array
        Array of actual ages.
    predicted_age : np.array
        Array of predicted ages.
    actual_age_test : np.array, optional
        Array of actual test ages. If not provided, `actual_age` is used.
    predicted_age_test : np.array, optional
        Array of predicted test ages. If not provided, `predicted_age` is used.

    Returns
    -------
    corrected_age_gap : np.array
        Array of corrected age gaps.
    corrected_predicted_age : np.array
        Array of corrected predicted ages.
    """
    mdl = LinearRegression()
    mdl.fit(actual_age.reshape(-1, 1), predicted_age.reshape(-1, 1))
    actual_age_test = (
        actual_age.reshape(-1, 1)
        if actual_age_test is None
        else actual_age_test.reshape(-1, 1)
    )
    predicted_age_test = (
        predicted_age.reshape(-1, 1)
        if predicted_age_test is None
        else predicted_age_test.reshape(-1, 1)
    )
    corrected_predicted_age = predicted_age_test + (
        actual_age_test - mdl.predict(actual_age_test)
    )
    corrected_age_gap = corrected_predicted_age - actual_age_test
    return corrected_age_gap.reshape(-1), corrected_predicted_age.reshape(-1)


def pearson_correlation_cv(y_true, y_pred, n_splits=10):
    """Compute Pearson correlation for cross-validated predictions."""
    r = []
    y_true = np.array_split(y_true, n_splits)
    y_pred = np.array_split(y_pred, n_splits)
    for i in range(n_splits):
        r.append(correlation_score(y_true[i], y_pred[i]))
    return np.mean(r), np.std(r)


def compute_true_pred_corr(true, pred, score_funcs, n_splits=10):
    """ "Compute correlation between true and predicted values."""
    true_splits = np.array_split(true, n_splits)
    pred_splits = np.array_split(pred, n_splits)
    corr = [
        {
            scorer: score_funcs[scorer](true_splits[i], pred_splits[i])
            for scorer in score_funcs
        }
        for i in range(n_splits)
    ]
    return corr

In [ ]:
def prepare_df():
    pred_scores = defaultdict(dict)
    for dset in DATASETS:
        pred_scores[dset] = {}
        for cont in CONTRASTS:
            cont = cont.replace(" ", "_").lower()
            try:
                pred_scores[dset][cont] = glob(
                    op.join(
                        ABS_PATH,
                        f"validation/3_prediction/3.1_brain_age_prediction/results/{dset}/age_{cont}.pkl",
                    )
                )[0]
            except:
                pass

    test_scores = pd.DataFrame()
    pred_true_holdout = {}
    for dset in pred_scores:
        tmp_gap_holdout = {}
        for i, task in enumerate(pred_scores[dset]):
            pred = pickle.load(open(pred_scores[dset][task], "rb"))
            if i == 0:
                print(
                    f"Dataset: {dset}, n = {pred[0]["y_true"].shape[0] + pred[0]["y_true_holdout"].shape[0]}"
                )

            # Create a default dictionary with default "pred", "true", and "gap" lists
            tmp_pred_true_gap_holdout = {
                "gap_corr": [],
                "pred_corr": [],
                "idx": [],
            }
            cv_corr_holdout_ = []
            for cv_res in pred:
                # Save chronological and brain age for each task.
                y_pred_all = cv_res["y_pred_all"]
                y_true_all = cv_res["y_true_all"]

                # Holdout set.
                y_true_holdout = cv_res["y_true_holdout"]
                y_pred_holdout = cv_res["y_pred_holdout"]
                gap_corr, pred_corr = correct_braingap(
                    y_true_all, y_pred_all, y_true_holdout, y_pred_holdout
                )
                cv_corr_holdout_ = cv_corr_holdout_ + compute_true_pred_corr(
                    y_true_holdout, pred_corr, correlation_score, n_splits=1
                )
                tmp_pred_true_gap_holdout["gap_corr"].append(gap_corr)
                tmp_pred_true_gap_holdout["pred_corr"].append(pred_corr)
                tmp_pred_true_gap_holdout["idx"].append(cv_res["y_holdout_idx"])

                tmp = pd.DataFrame(cv_corr_holdout_)
                tmp["Contrast"] = task.replace("_", "\n").upper()
                tmp["Actual"] = "Actual" if "predicted" not in dset else "Predicted"
                test_scores = pd.concat(
                    [
                        test_scores,
                        tmp,
                    ],
                    axis=0,
                )
            tmp_gap_holdout.update({task: tmp_pred_true_gap_holdout})
        pred_true_holdout.update({dset: tmp_gap_holdout})
    return test_scores, pred_true_holdout


def plot_results(test_scores):
    plt.figure(figsize=(10, 5), dpi=300)
    sns.pointplot(
        data=test_scores,
        x="Contrast",
        y="Correlation",
        hue="Actual",
        errorbar="sd",
        linestyles="",
        palette=PALETTE,
    )
    # New code to make axes lines thinner
    plt.ylim((0.2, 0.8))  # Set y-axis limits
    sns.despine(offset=10, trim=True)
    plt.ylabel("Correlation")
    plt.legend()
    plt.savefig(op.join(FIG_DIR, "age_prediction.pdf"), bbox_inches="tight")


test_scores, pred_true_holdout = prepare_df()
plot_results()

In [ ]:
def compare_prediction_scores(test_scores):
    """Compare prediction scores using permutated paired t-test (n_permutation = 1000)."""

    def statistic(x, y):
        return ttest_rel(x, y).statistic

    # Permutated paired t-test
    # Predicted maps vs. Actual Maps
    # Predicted maps vs. Rest
    post_hoc = []
    for pred_cont in np.unique(
        test_scores[test_scores["Actual"] == "Predicted"]["Contrast"]
    ):
        pred_y = test_scores[
            (test_scores["Actual"] == "Predicted")
            & (test_scores["Contrast"] == pred_cont)
        ]["Correlation"].values
        for comp_cont in np.unique(
            test_scores[test_scores["Actual"] == "Actual"]["Contrast"]
        ):
            comp_y = test_scores[
                (test_scores["Actual"] == "Actual")
                & (test_scores["Contrast"] == comp_cont)
            ]["Correlation"].values
            # stat, p_val = ttest_rel(pred_y, comp_y)
            res = permutation_test(
                (pred_y, comp_y),
                statistic,
                vectorized=False,
                permutation_type="samples",
                alternative="two-sided",
                random_state=42,
                n_resamples=1000,
            )
            stat, p_val = res.statistic, res.pvalue
            post_hoc.append(
                {
                    "Predicted Contrast": pred_cont,
                    "Actual Contrast": comp_cont,
                    "t": stat,
                    "p": p_val,
                }
            )
    # Compare REST > Actual EMOTION FACES-SHAPES
    res = permutation_test(
        (
            test_scores[test_scores["Contrast"] == "REST"]["Correlation"].values,
            test_scores[
                (test_scores["Contrast"] == "EMOTION\nFACES-SHAPES")
                & (test_scores["Actual"] == "Actual")
            ]["Correlation"].values,
        ),
        statistic,
        vectorized=False,
        permutation_type="samples",
        alternative="two-sided",
        random_state=42,
        n_resamples=1000,
    )
    stat, p_val = res.statistic, res.pvalue
    post_hoc.append(
        {
            "Predicted Contrast": "REST",
            "Actual Contrast": "EMOTION\nFACES-SHAPES",
            "t": stat,
            "p": p_val,
        }
    )
    pd.DataFrame(post_hoc).to_csv(
        op.join(FIG_DIR, "age_prediction_comparisons.csv"), index=False
    )


compare_prediction_scores(test_scores)

In [ ]:
def associate_bag(pred_true_holdout):
    """Associate brain age gap (BAG) scores with fluid intelligence, strength, and overall health."""

    def statistic(x, y):
        return spearmanr(x, y).statistic

    for tar in ["fluid", "strength", "overall_health"]:
        print(f"\nTarget: {tar}\n")
        corr_df = pd.DataFrame()
        for dset in ["ukb", "ukb_pred"]:
            cont_maps = "Predicted" if "pred" in dset else "Actual"
            for i, task in enumerate(pred_true_holdout[dset].keys()):
                # Prepare Target
                holdout_df = ALL_TARGETS.iloc[
                    np.concatenate(pred_true_holdout[dset][task]["idx"])
                ]
                nan_idx = np.isnan(holdout_df[tar].values)
                target = holdout_df[tar].values[~nan_idx]
                if i == 0:
                    print(f"Dataset: {dset}, Target: {tar}, n = {len(target)}")
                # Run Comparisons
                bag = np.concatenate(pred_true_holdout[dset][task]["gap_corr"])[
                    ~nan_idx
                ]
                res = permutation_test(
                    (bag, target),
                    statistic,
                    vectorized=False,
                    permutation_type="pairings",
                    alternative="two-sided",
                    random_state=42,
                    n_resamples=1000,
                )
                r, pvalue = res.statistic, res.pvalue
                corr_df = pd.concat(
                    [
                        corr_df,
                        pd.DataFrame(
                            {
                                "Task": task,
                                "Maps": cont_maps,
                                "r": r,
                                "p": pvalue,
                            },
                            index=[0],
                        ),
                    ]
                )
        corr_df.to_csv(op.join(FIG_DIR, f"bag_{tar}_assoc.csv"), index=False)
        print(corr_df)


associate_bag(pred_true_holdout)